In [1]:
!pip install -q torch transformers ninja tokenizers rwkv pynvml huggingface_hub smolagents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 95.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 77.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 17.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 10.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 7.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.0/410.0 kB 752.5 kB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━

In [3]:
# tools
!pip install -q ddgs wikipedia-api markdownify readability-lxml

In [4]:
import os
import re
import copy
import time
import gc

import torch
import huggingface_hub
from transformers import AutoTokenizer

from smolagents.agents import EMPTY_PROMPT_TEMPLATES
from smolagents.models import Model, ChatMessage, MessageRole
from smolagents.monitoring import TokenUsage
from smolagents.tools import Tool
from smolagents.default_tools import (
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    VisitWebpageTool,
    WikipediaSearchTool
)
from smolagents import CodeAgent

from IPython.display import clear_output

## Custom tools

In [5]:
class CustomDuckDuckGoSearchTool(DuckDuckGoSearchTool):
    name = "web_search"
    description = "Performs a web search for a query and returns a list of the top search results formatted as markdown with page titles and urls."
    inputs = {"query": {"type": "string", "description": "The search query to perform."}}
    output_type = "string"

    def __init__(self, max_results: int = 10, rate_limit: float | None = 1.0, backend: str = "auto", **kwargs):
        super().__init__(max_results=max_results, rate_limit=rate_limit, **kwargs)
        self.backend = backend # Add "backend" as new parameter

    def forward(self, query: str) -> str:
        self._enforce_rate_limit()
        results = self.ddgs.text(
            query=query,
            max_results=self.max_results,
            backend=self.backend) # there you go
        
        if len(results) == 0:
            raise Exception("No results found! Try a less restrictive/shorter query.")
        
        postprocessed_results = [
            f"{i+1}. [{result['title']}]({result['href']})\nBody: {result['body']}"
            for i, result in enumerate(results)
        ]

        return "## Search Results\n" + "\n".join(postprocessed_results)

In [57]:
class CleanVisitWebpageTool(VisitWebpageTool):
    """
    Improved version of VisitWebpageTool that extracts and cleans the main content
    of a webpage for efficient LLM consumption.
    """

    name = "visit_webpage"
    description = (
        "Visits a webpage at the given url and reads its content as a markdown string. Use this to browse webpages."
    )
    inputs = {
        "url": {
            "type": "string",
            "description": "The url of the webpage to visit.",
        }
    }
    output_type = "string"

    def __init__(self, max_output_length: int = 4000, clean_level: int = 2):
        super().__init__(max_output_length=max_output_length)
        self.clean_level = clean_level  # 0 = raw, 1 = cleaned, 2 = aggressively cleaned

    def _extract_main_content(self, html: str) -> tuple[str, str]:
        """
        Extracts the main content (article body) from the HTML using readability-lxml.
        Returns (title, main_html)
        """
        try:
            from readability.readability import Document
            doc = Document(html)
            title = doc.title() or ""
            main_html = doc.summary() or html
            return title, main_html
        except Exception:
            # Fallback: return raw HTML if readability fails
            return "", html

    def _clean_markdown(self, md: str) -> str:
        """
        Removes common noise, legal text, and repeated fluff from markdown text.
        """
        import re

        # Remove excessive blank lines
        md = re.sub(r'\n{2,}', '\n', md)

        junk_patterns = [
            r"(?i)(privacy policy|terms of use|cookie|subscribe|©|copyright).*",
            r"^\s*Share\s*$",
            r"(?i)related (articles|posts)",
            r"(?i)follow us",
            r"(?i)back to top",
        ]
        for pattern in junk_patterns:
            md = re.sub(pattern, '', md)

        # Remove short or generic markdown links (like [Home](...))
        md = re.sub(r'\[([^\]]{1,15})\]\([^)]+\)', r'\1', md)

        # Collapse multiple spaces and trim
        md = re.sub(r'[ \t]{2,}', ' ', md)
        return md.strip()

    def forward(self, url: str) -> str:
        try:
            import re
            import requests
            from markdownify import markdownify
            from requests.exceptions import RequestException
        except ImportError as e:
            raise ImportError(
                "You must install `markdownify`, `readability-lxml`, and `requests` "
                "to run CleanVisitWebpageTool: "
                "`pip install markdownify readability-lxml requests`."
            ) from e

        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()

            html = response.text
            title, main_html = self._extract_main_content(html)

            markdown_content = markdownify(main_html, heading_style="ATX").strip()

            # Apply cleaning if requested
            if self.clean_level >= 1:
                markdown_content = self._clean_markdown(markdown_content)

            # Add title as a header
            if title:
                markdown_content = f"# {title}\n{markdown_content}"

            # Truncate if too long
            markdown_content = self._truncate_content(markdown_content, self.max_output_length)

            return markdown_content

        except requests.exceptions.Timeout:
            return "The request timed out. Please try again later or check the URL."
        except RequestException as e:
            return f"Error fetching the webpage: {str(e)}"
        except Exception as e:
            return f"An unexpected error occurred: {str(e)}"

In [7]:
os.environ["RWKV_V7_ON"] = '1'
os.environ["RWKV_JIT_ON"] = '1'
os.environ["RWKV_CUDA_ON"] = '1' # if '1' then use CUDA kernel for seq mode (much faster)

from rwkv.model import RWKV
from rwkv.utils import PIPELINE, PIPELINE_ARGS

Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv_cuda...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv_cuda/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv_cuda...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/4] c++ -MMD -MF wrapper.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/wrapper.cpp -o wrapper.o 
[2/4] c++ -MMD -MF gemm_fp16_cublas.o.d -DTORCH_EXTENSION_NAME=wkv_cuda -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/incl

Loading extension module wkv_cuda...
Using /root/.cache/torch_extensions/py311_cu124 as PyTorch extensions root...
Creating extension directory /root/.cache/torch_extensions/py311_cu124/wkv7s...
Detected CUDA files, patching ldflags
Emitting ninja build file /root/.cache/torch_extensions/py311_cu124/wkv7s/build.ninja...
/usr/local/lib/python3.11/dist-packages/torch/utils/cpp_extension.py:2059: UserWarning: TORCH_CUDA_ARCH_LIST is not set, all archs for visible cards are included for compilation. 
If this is not desired, please set os.environ['TORCH_CUDA_ARCH_LIST'].
  warnings.warn(
Building extension module wkv7s...
Allowing ninja to set a default number of workers... (overridable by setting the environment variable MAX_JOBS=N)


[1/3] c++ -MMD -MF rwkv7_op.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/local/lib/python3.11/dist-packages/torch/include/TH -isystem /usr/local/lib/python3.11/dist-packages/torch/include/THC -isystem /usr/local/cuda/include -isystem /usr/include/python3.11 -D_GLIBCXX_USE_CXX11_ABI=0 -fPIC -std=c++17 -c /usr/local/lib/python3.11/dist-packages/rwkv/cuda/rwkv7_op.cpp -o rwkv7_op.o 
[2/3] /usr/local/cuda/bin/nvcc --generate-dependencies-with-compile --dependency-output rwkv7.cuda.o.d -DTORCH_EXTENSION_NAME=wkv7s -DTORCH_API_INCLUDE_EXTENSION_H -DPYBIND11_COMPILER_TYPE=\"_gcc\" -DPYBIND11_STDLIB=\"_libstdcpp\" -DPYBIND11_BUILD_ABI=\"_cxxabi1011\" -isystem /usr/local/lib/python3.11/dist-packages/torch/include

Loading extension module wkv7s...


In [8]:
# model_title = "rwkv7-g1a-2.9b-20250924-ctx4096"
# model_title = "rwkv7-g0a2-7.2b-20251005-ctx4096"
model_title = "rwkv7-g0a3-7.2b-20251029-ctx8192"
model_path = huggingface_hub.hf_hub_download(repo_id="BlinkDL/rwkv7-g1", filename=f"{model_title}.pth")
rwkv_model = RWKV(model=model_path.replace('.pth',''), strategy='cuda fp16')
rwkv_pipeline = PIPELINE(rwkv_model, "rwkv_vocab_v20230424")

rwkv7-g0a3-7.2b-20251029-ctx8192.pth:   0%|          | 0.00/14.4G [00:00<?, ?B/s]

Loading /root/.cache/huggingface/hub/models--BlinkDL--rwkv7-g1/snapshots/d1c8bf401e7f41b92101c3e9f3a9a54094304aed/rwkv7-g0a3-7.2b-20251029-ctx8192 (cuda fp16)



In [58]:
class RWKVModel(Model):
    """
    A subclass of Model for RWKV7 inference, adapted from the experimental notebook.
    """
    def __init__(
        self,
        model,
        pipeline,
        flatten_messages_as_text: bool = False,
        tool_name_key: str = "name",
        tool_arguments_key: str = "arguments",
        model_id: str | None = None,
        **kwargs,
    ):    
        # Setup RWKV
        self.CTX_LIMIT = 6000
        self.PENALTY_DECAY = 0.996
        self._model = model
        self._pipeline = pipeline

        # Store state of the model
        self.state = None
        self.__reset_state = True
        
        # Only for chat template
        # TODO: Get rid of it...
        self.tokenizer = AutoTokenizer.from_pretrained('fla-hub/rwkv7-2.9B-g1', trust_remote_code=True)
        
        super().__init__(flatten_messages_as_text=True, model_id="model_id", **kwargs)

    def reset_state(self) -> None:
        self.state = None
    
    def generate(
        self,
        messages: list[ChatMessage],
        stop_sequences: list[str] | None = None,
        response_format: dict[str, str] | None = None,
        tools_to_call_from: list[Tool] | None = None,
        **kwargs,
    ) -> ChatMessage:
        """Process the input messages and return the model's response."""
        if self.__reset_state:
            print("Reseting state...")
            self.reset_state()
        else:
            print("Reuse previous state...")
        
        completion_kwargs = self._prepare_completion_kwargs(
            messages=messages,
            flatten_messages_as_text=self.flatten_messages_as_text,
            stop_sequences=stop_sequences,
            tools_to_call_from=tools_to_call_from,
            **kwargs,
        )

        # print("Completion kwargs", completion_kwargs)

        messages = completion_kwargs.pop("messages")
        prepared_stop_sequences = completion_kwargs.pop("stop", [])
        tools = completion_kwargs.pop("tools", None)
        completion_kwargs.pop("tool_choice", None)

        prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            tools=tools,
            add_generation_prompt=True,
            enable_thinking=False
        )

        # Remove <|rwkv_tokenizer_end_of_text|>
        prompt = prompt.replace("<|rwkv_tokenizer_end_of_text|>", "").strip()

        print(f"Input prompt: ", prompt)

        output = self.run_rwkv(prompt)
        # Test string
        # output = """Thought: I will use the `math` module to compute the value of exp(10) and then add it to 2.\n<code>\nimport math\nresult = 2 + math.exp(10)\nprint(result)\n</code>"""

        output_text = output.strip()

        # print(f"Output: ", output_text)
        
        return ChatMessage(
            role=MessageRole.ASSISTANT,
            content=output_text,
            raw={"out": output_text},
            token_usage=TokenUsage(
                input_tokens=len(prompt.split()),
                output_tokens=len(output_text.split()),
            ),
        )

    def run_rwkv(
        self,
        ctx,
        token_count=500,
        temperature=1.0,
        top_p=0.7,
        presencePenalty=0.1,
        countPenalty=0.1,
    ):
        # TODO: Use generate(), or get rid of PIPELINE_ARGS
        args = PIPELINE_ARGS(
            temperature=max(0.2, float(temperature)),
            top_p=float(top_p),
            alpha_frequency=countPenalty,
            alpha_presence=presencePenalty,
            token_ban=[],        # ban the generation of some tokens
            token_stop=[261]       # stop generation whenever you see any token here
        )
    
        ctx = ctx.strip()
        all_tokens = []
        out_last = 0
        out_str = ''
        occurrence = {}
    
        for i in range(int(token_count)):
            input_ids = self._pipeline.encode(ctx)
            input_ids = input_ids[-self.CTX_LIMIT:] if i == 0 else [token]
            
            out, self.state = self._model.forward(input_ids, self.state)
    
            for n in occurrence:
                out[n] -= (args.alpha_presence + occurrence[n] * args.alpha_frequency)
    
            token = self._pipeline.sample_logits(
                out,
                temperature=args.temperature,
                top_p=args.top_p)
            if token in args.token_stop:
                break
    
            all_tokens.append(token)
    
            for xxx in occurrence:
                occurrence[xxx] *= self.PENALTY_DECAY
    
            ttt = self._pipeline.decode([token])
            www = 1
            if ttt in ' \t0123456789':
                www = 0
            if token not in occurrence:
                occurrence[token] = www
            else:
                occurrence[token] += www
    
            tmp = self._pipeline.decode(all_tokens[out_last:])
            if '\ufffd' not in tmp:
                out_str += tmp
                out_last = i + 1

                if out_str.strip().endswith("\n\n"):
                    break
    
        # Cleanup and timing
        del out
        gc.collect()
        torch.cuda.empty_cache()
        
        return out_str.strip()

    def prepare_messages(self, messages: list[dict[str]]) -> list[dict[str]]:
        new_messages = []
    
        for m in messages:
            content = m.get("content", "")
            if content.strip():
                content = re.sub(r'\n+', '\n', content)
                content = re.sub(r'\n+$', '', content)
                m["content"] = content
                new_messages.append(m)

        return new_messages

In [59]:
model = RWKVModel(model=rwkv_model, pipeline=rwkv_pipeline)

In [64]:
SYSTEM_PROMPT = '''You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '{{code_block_opening_tag}}', and closed with '{{code_block_closing_tag}}'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.
Here are a few examples using notional tools:
---
User: New task:
What is the result of the following operation: 5 + 3 + 1294.678?

Assistant:<think>
</think>Thought: I will use Python code to compute the result of the operation and then return the final answer using the `final_answer` tool.
{{code_block_opening_tag}}
result = 5 + 3 + 1294.678
final_answer(result)
{{code_block_closing_tag}}

---
User: New task:
What is the height of the Tower of Pisa?

Assistant:<think>
</think>Thought: I will use the tool `web_search` and analyze results get the of height of the tower.
{{code_block_opening_tag}}
search_result=web_search(query="height of the tower of pisa")
print(search_result)
{{code_block_closing_tag}}

User:
Observation:
## Search Results
1. [HOW Tall is the Leaning Tower of Pisa?](https://leaningtowerpisa.com/facts/how-tall-leaning-tower-pisa)
Body: Who built the Tower needed to make several corrections along the way but the target height of 100 arms was kept and reached with a compromise: measured from the foundation plan and not from the ground level. Did you know the Leaning Tower of Pisa is exactly 100 "Tuscan arms" tall? Often one needs to make compromises in order to achieve big goals.
2. [Leaning Tower of Pisa | History, Architecture, Foundation & Lean ...](https://www.britannica.com/topic/Leaning-Tower-of-Pisa)
Body: Leaning Tower of Pisa , medieval structure in Pisa , Italy, that is famous for the settling of its foundations, which caused it to lean 5.5 degrees (about 15 feet [4.5 metres]) from the perpendicular in the late 20th century. Extensive work was subsequently done to straighten the tower , and its lean
3. [Leaning Tower of Pisa - Wikipedia](https://en.wikipedia.org/wiki/Leaning_Tower_of_Pisa)
Body: The Leaning Tower of Pisa (Italian: torre pendente di Pisa [ˈtorre penˈdɛnte di ˈpiːza, - ˈ piːsa ] [1]), or simply the Tower of Pisa (torre di Pisa ), is the campanile, or freestanding bell tower , of Pisa Cathedral. It is known for its nearly four-degree lean, the result of an unstable foundation.
(truncated)

Assistant: <think>
</think>
Thought: I have found some relevant results. Now I must use `visit_webpage` tool to access contents of the "https://leaningtowerpisa.com/facts/how-tall-leaning-tower-pisa" web page
{{code_block_opening_tag}}
result = visit_webpage("https://leaningtowerpisa.com/facts/how-tall-leaning-tower-pisa")
print(result)
{{code_block_closing_tag}}

User:
Observation:
# HOW Tall is the Leaning Tower of Pisa? - Leaning Tower Pisa
This is probably one of the most asked questions... to the point that Google gives a straight answer: **57 meters**. 
However, you should not always believe what you are told and - just for this time - you should not trust even Google.
Why? 
...because **GOOGLE IS WRONG** on this one: the Leaning Tower of Pisa is **NOT** 57 meters tall...
(truncated)

Assistant: <think>
</think>
Thought: I have analyzed the contents of the web page from the above observation.
I have found that height of the tower is 57 meters. Now I will use final_answer() tool to output the answer.
{{code_block_opening_tag}}
final_answer("57 meters")
{{code_block_closing_tag}}
---
Above examples were using notional tools that might not exist for you.
On top of performing computations in the Python code snippets that you create, you only have access to these tools, behaving like regular python functions:
{{code_block_opening_tag}}
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{%- endfor -%}
{{code_block_closing_tag}}
Here are the rules you should always follow to solve your task:
1. Always provide a 'Thought:' sequence, and a '{{code_block_opening_tag}}' sequence ending with '{{code_block_closing_tag}}', else you will fail.
2. Use only variables that you have defined!
3. Use only tools that provided to you!
4. To complete the task you need to call ```final_answer()``` tool!
5. Always use the right arguments for the tools. DO NOT pass the arguments as a dict as in 'answer = wikipedia_search({'query': "What is the place where James Bond lives?"})', but use the arguments directly as in 'answer = wikipedia_search(query="What is the place where James Bond lives?")'.
6. Call a tool only when needed, and never re-do a tool call that you previously did with the exact same parameters.
7. Don't name any new variable with the same name as a tool: for instance don't name a variable 'final_answer'.
8. Never create any notional variables in our code, as having these in your logs will derail you from the true variables.
9. You can use imports in your code, but only from the following list of modules: {{authorized_imports}}
10. The state persists between code executions: so if in one step you've created variables or imported modules, these will all persist.
11. Don't give up! You're in charge of solving the task, not providing directions to solve it.
12. Alsways use ```{{code_block_opening_tag}}final_answer(){{code_block_closing_tag}}``` tool to complete the task!
---
Now Begin!'''

In [14]:
INITIAL_PLAN_PROMPT = '''You are a world expert at analyzing a situation to derive facts, and plan accordingly towards solving a task.
Below I will present you a task. You will need to 1. build a survey of facts known or needed to solve the task, then 2. make a plan of action to solve the task.
## 1. Facts survey
You will build a comprehensive preparatory survey of which facts we have at our disposal and which ones we still need.
These "facts" will typically be specific names, dates, values, etc. Your answer should use the below headings:
### 1.1. Facts given in the task
List here the specific facts given in the task that could help you (there might be nothing here).
### 1.2. Facts to look up
List here any facts that we may need to look up.
Also list where to find each of these, for instance a website, a file... - maybe the task contains some sources that you should re-use here.
### 1.3. Facts to derive
List here anything that we want to derive from the above by logical reasoning, for instance computation or simulation.
Don't make any assumptions. For each item, provide a thorough reasoning. Do not add anything else on top of three headings above.
## 2. Plan
Then for the given task, develop a step-by-step high-level plan taking into account the above inputs and list of facts.
This plan should involve individual tasks based on the available tools, that if executed correctly will yield the correct answer.
Do not skip steps, do not add any superfluous steps. Only write the high-level plan, DO NOT DETAIL INDIVIDUAL TOOL CALLS.
After writing the final step of the plan, write the '<end_plan>' tag and stop there.
You can leverage these tools, behaving like regular python functions:
<code>
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{% endfor %}
</code>
Here are the rules you should always follow to solve your task:
1. You MUST provide only high-level plan.
2. You MUST not call tools or write any code at this stage.
---
Now begin! Here is your task:
```
{{task}}
```
First in part 1, write the facts survey, then in part 2, write your plan.'''

In [15]:
UPDATE_PLAN_PRE = '''You are a world expert at analyzing a situation, and plan accordingly towards solving a task.
You have been given the following task:
```
{{task}}
```
Below you will find a history of attempts made to solve this task.
You will first have to produce a survey of known and unknown facts, then propose a step-by-step high-level plan to solve the task.
If the previous tries so far have met some success, your updated plan can build on these results.
If you are stalled, you can make a completely new plan starting from scratch.
Find the task and history below:'''

UPDATE_PLAN_POST = '''Now write your updated facts below, taking into account the above history:
## 1. Updated facts survey
### 1.1. Facts given in the task
### 1.2. Facts that we have learned
### 1.3. Facts still to look up
### 1.4. Facts still to derive
Then write a step-by-step high-level plan to solve the task above.
## 2. Plan
### 2. 1. ...
Etc.
This plan should involve individual tasks based on the available tools, that if executed correctly will yield the correct answer.
Beware that you have {remaining_steps} steps remaining.
Do not skip steps, do not add any superfluous steps. Only write the high-level plan, DO NOT DETAIL INDIVIDUAL TOOL CALLS.
After writing the final step of the plan, write the '<end_plan>' tag and stop there.
You can leverage these tools, behaving like regular python functions:
<code>
{%- for tool in tools.values() %}
{{ tool.to_code_prompt() }}
{% endfor %}
</code>
Now write your updated facts survey below, then your new plan.'''

In [16]:
FINAL_PRE = '''An agent tried to answer a user query but it got stuck and failed to do so. You are tasked with providing an answer instead. Here is the agent's memory:'''
FINAL_POST = '''Based on the above, please provide an answer to the following user task:
{{task}}'''

In [65]:
RWKV_PROMPT_TEMPLATES = EMPTY_PROMPT_TEMPLATES
RWKV_PROMPT_TEMPLATES["system_prompt"] = SYSTEM_PROMPT

# planning only
RWKV_PROMPT_TEMPLATES["planning"]["initial_plan"] = INITIAL_PLAN_PROMPT
RWKV_PROMPT_TEMPLATES["planning"]["update_plan_pre_messages"] = UPDATE_PLAN_PRE
RWKV_PROMPT_TEMPLATES["planning"]["update_plan_post_messages"] = UPDATE_PLAN_POST

# error?
RWKV_PROMPT_TEMPLATES["final_answer"]["pre_messages"] = FINAL_PRE
RWKV_PROMPT_TEMPLATES["final_answer"]["post_messages"] = FINAL_POST

In [62]:
agent = CodeAgent(
    tools=[
        CustomDuckDuckGoSearchTool(max_results=5, rate_limit=1.0, backend="google"),
        CleanVisitWebpageTool(),
        FinalAnswerTool(),
    ],
    model=model,
    prompt_templates=RWKV_PROMPT_TEMPLATES,
    planning_interval=2,
    verbosity_level=2)

In [55]:
model.reset_state()
agent.run("What is the result of 2 + exp(10) + sin(pi/2)?", max_steps=2)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the result of 2 + exp(10) + sin(pi/2)?                                                                  │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

Reseting state...
Input prompt:  User: You are a world expert at analyzing a situation to derive facts, and plan accordingly towards solving a task.
Below I will present you a task. You will need to 1. build a survey of facts known or needed to solve the task, then 2. make a plan of action to solve the task.
## 1. Facts survey
You will build a comprehensive preparatory survey of which facts we have at our disposal and which ones we still need.
These "facts" will typically be specific names, dates, values, etc. Your answer should use the below headings:
### 1.1. Facts given in the task
List here the specific facts given in the task that could help you (there might be nothing here).
### 1.2. Facts to look up
List here any facts that we may need to look up.
Also list where to find each of these, for instance a website, a file... - maybe the task contains some sources that you should re-use here.
### 1.3. Facts to derive
List here anything that we want to derive from the above by logical r

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1. Facts Survey
#### Facts given in the task:
- The expression is `2 + exp(10) + sin(pi/2)`.
- The task is to compute the result of this expression.
- No additional context or constraints are provided.
#### Facts to look up:
- The value of `exp(10)`, which is the exponential function `e^10` where `e` is Euler's number (approximately 
2.71828).
- The value of `sin(pi/2)`, which is the sine of pi divided by 2. Since `pi` is approximately 3.14159, `pi/2` is 
1.5708 radians, and `sin(pi/2) = 1`.
#### Facts to derive:
- None. The task is straightforward with given inputs, and no further computation or derivation is needed.
### 2. Plan
#### Step-by-step high-level plan:
1. Calculate the value of `exp(10)` by evaluating the exponential function at 10.
2. Calculate the value of `sin(pi/2)` by evaluating the sine function at pi/2 radians.
3. Add the results from step 1 and step 2 to compute the final answer.
4. Format the final answer appropriately for the task.
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reseting state...
Input prompt:  System: You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will use the `math` module in Python to compute the values of `exp(10)` and `sin(pi/2)`.                
<code>                                                                                                             
import math                                                                                                        
# Calculate exp(10)                                                                                                
exp_value = math.exp(10)                                                                                           
# Calculate sin(pi/2)                                                                                              
sin_value = math.sin(math.pi / 2)                                                                                  
# Calculate the final result                                                                                       
final_answer = exp_value + sin_value                                                                               
# Format the final answer                                                                                          
formatted_answer = f"{exp_value:.10f} + {sin_value:.10f} = {final_answer:.10f}"                                    
</code>                                                                                                            
Thought: I have calculated the values of `exp(10)` and `sin(pi/2)`, and added them together to get the final       
answer. The final answer is formatted to 10 decimal places for precision.                                          
<code>                                                                                                             
final_answer(formatted_answer)                                                                                     
</code>                                                                                                            

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import math                                                                                                      
  # Calculate exp(10)                                                                                              
  exp_value = math.exp(10)                                                                                         
  # Calculate sin(pi/2)                                                                                            
  sin_value = math.sin(math.pi / 2)                                                                                
  # Calculate the final result                                                                                     
  final_answer_variable = exp_value + sin_value                                                                    
  # Format the final answer                                                                                        
  formatted_answer = f"{exp_value:.10f} + {sin_value:.10f} = {final_answer_variable:.10f}"                         
                                                                                                                   
  final_answer(formatted_answer)                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 22026.4657948067 + 1.0000000000 = 22027.4657948067

[Step 1: Duration 17.10 seconds| Input tokens: 1,223 | Output tokens: 90]

'22026.4657948067 + 1.0000000000 = 22027.4657948067'

In [66]:
model.reset_state()
agent.run("Search for the most popular programming language for web in 2025", max_steps=4)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Search for the most popular programming language for web in 2025                                                │
│                                                                                                                 │
╰─ RWKVModel - model_id ──────────────────────────────────────────────────────────────────────────────────────────╯

Reseting state...
Input prompt:  User: You are a world expert at analyzing a situation to derive facts, and plan accordingly towards solving a task.
Below I will present you a task. You will need to 1. build a survey of facts known or needed to solve the task, then 2. make a plan of action to solve the task.
## 1. Facts survey
You will build a comprehensive preparatory survey of which facts we have at our disposal and which ones we still need.
These "facts" will typically be specific names, dates, values, etc. Your answer should use the below headings:
### 1.1. Facts given in the task
List here the specific facts given in the task that could help you (there might be nothing here).
### 1.2. Facts to look up
List here any facts that we may need to look up.
Also list where to find each of these, for instance a website, a file... - maybe the task contains some sources that you should re-use here.
### 1.3. Facts to derive
List here anything that we want to derive from the above by logical r

────────────────────────────────────────────────── Initial plan ───────────────────────────────────────────────────
Here are the facts I know and the plan of action that I will follow to solve the task:
```
### 1. Facts survey
#### Facts given in the task
- Task: Search for the most popular programming language for web in 2025
#### Facts to look up
- **Current top programming languages for web development**: Based on recent trends (e.g., Stack Overflow Developer
Survey 2023, GitHub Octoverse 2023), JavaScript is the most popular language for web development.
- **Market trends for web development**: JavaScript has maintained dominance due to frameworks like React, Vue, and
Angular. However, there is growing interest in newer languages like TypeScript, Python, and Rust for specific use 
cases.
- **Emerging technologies**: Frameworks like SvelteKit and Astro are gaining traction for web development, but they
are often built on JavaScript.
- **Industry reports**: Sources like RedMonk Programming Language Rankings or Stack Overflow Developer Survey can 
provide insights into future trends.
#### Facts to derive
- **Analysis of current trends**: Evaluate if JavaScript's dominance is changing or if other languages are gaining 
share.
- **Assessment of new frameworks**: Determine if frameworks like SvelteKit or Astro are shifting the landscape.
- **Consideration of other languages**: Explore languages like TypeScript, Python (Django/Flask), or Rust 
(WebAssembly) for web development.
- **Conclusion based on data**: Synthesize findings to identify the most popular language for web development in 
2025.
### 2. Plan
1. **Conduct a web search** for "most popular programming language for web in 2025" to gather initial data and 
recent trends.
2. **Visit websites** such as Stack Overflow Developer Survey, RedMonk Programming Language Rankings, and GitHub 
Octoverse to access detailed reports on programming language popularity.
3. **Visit websites** for frameworks like SvelteKit and Astro to understand their market share and adoption rates 
in web development.
4. **Derive conclusions** by analyzing the data collected, focusing on trends and expert opinions to identify the 
most popular language for web development in 2025.
```

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reseting state...
Input prompt:  System: You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 4 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Reseting state...
Input prompt:  System: You are an expert assistant who can solve any task using code blobs.
You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.
At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '<code>', and closed with '</code>'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final

Error in generating model output:
The following operation failed in the TorchScript interpreter.
Traceback of TorchScript (most recent call last):
RuntimeError: CUDA out of memory. Tried to allocate 34.00 MiB. GPU 0 has a total capacity of 15.89 GiB of which 
23.12 MiB is free. Process 2846 has 15.86 GiB memory in use. Of the allocated memory 15.45 GiB is allocated by 
PyTorch, and 130.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try 
setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory 
Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

[Step 4: Duration 5.99 seconds]

AgentGenerationError: Error in generating model output:
The following operation failed in the TorchScript interpreter.
Traceback of TorchScript (most recent call last):
RuntimeError: CUDA out of memory. Tried to allocate 34.00 MiB. GPU 0 has a total capacity of 15.89 GiB of which 23.12 MiB is free. Process 2846 has 15.86 GiB memory in use. Of the allocated memory 15.45 GiB is allocated by PyTorch, and 130.12 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)
